In [ ]:
from scipy.stats import spearmanr
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
# import import_ipynb
from scipy.interpolate import PchipInterpolator
import itertools



def J_univariante(X, tau, corte):
    def distancia(p1, p2):
        return np.linalg.norm(np.array(p2) - np.array(p1))
    X = np.array(X)
    x1 = X[tau:]
    y1 = X[:-tau]
    ff1 = np.angle(np.fft.rfft(x1))
    ff2 = np.angle(np.fft.rfft(y1))

    vectores = []
    for i in range(len(ff1) - 1):
        p1 = [ff1[i], ff2[i]]
        p2 = [ff1[i + 1], ff2[i + 1]]
        cuadrante = [
            [p2[0] - p1[0], p2[1] - p1[1]],
            [p2[0] - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
        ]
        distancias = np.array([distancia(p1, c) for c in cuadrante])
        p2 = cuadrante[np.argmin(distancias)]
        vectores.append([p2[0] - p1[0], p2[1] - p1[1]])

    vectores = np.array(vectores)
    norms = np.linalg.norm(vectores, axis=1, keepdims=True)
    v_norm = np.where(norms == 0, vectores, vectores / norms)
    
    angulos = np.arccos(np.clip(np.einsum('ij,ij->i', v_norm[:-1], v_norm[1:]), -1.0, 1.0))
    cruces = np.cross(v_norm[:-1], v_norm[1:])
    angulos = np.where(cruces > 0, np.pi - angulos, angulos)
    angulos = np.where((cruces == 0) & (angulos < 0), np.pi, angulos)
    angulos = np.where(cruces < 0, angulos + np.pi, angulos)

    e = np.exp(angulos * 1j)
    e1 = np.sum(e) / len(angulos)
    J = 1.0 - np.abs(e1.real)
    
    return J

def J_bivariante(X, Y, corte):
    def distancia(p1, p2):
        return np.linalg.norm(np.array(p2) - np.array(p1))
    X = np.array(X)
    x1 = X[:]
    y1 = Y[:]
    ff1 = np.angle(np.fft.rfft(x1))
    ff2 = np.angle(np.fft.rfft(y1))

    vectores = []
    for i in range(len(ff1) - 1):
        p1 = [ff1[i], ff2[i]]
        p2 = [ff1[i + 1], ff2[i + 1]]
        cuadrante = [
            [p2[0] - p1[0], p2[1] - p1[1]],
            [p2[0] - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
        ]
        distancias = np.array([distancia(p1, c) for c in cuadrante])
        p2 = cuadrante[np.argmin(distancias)]
        vectores.append([p2[0] - p1[0], p2[1] - p1[1]])

    vectores = np.array(vectores)
    norms = np.linalg.norm(vectores, axis=1, keepdims=True)
    v_norm = np.where(norms == 0, vectores, vectores / norms)
    
    angulos = np.arccos(np.clip(np.einsum('ij,ij->i', v_norm[:-1], v_norm[1:]), -1.0, 1.0))
    cruces = np.cross(v_norm[:-1], v_norm[1:])
    angulos = np.where(cruces > 0, np.pi - angulos, angulos)
    angulos = np.where((cruces == 0) & (angulos < 0), np.pi, angulos)
    angulos = np.where(cruces < 0, angulos + np.pi, angulos)

    e = np.exp(angulos * 1j)
    e1 = np.sum(e) / len(angulos)
    J = 1.0 - np.abs(e1.real)
    
    return J

def interpolador(subject, method, size):
    # data = np.array([int(line.strip()) for line in subject.to_numpy()])  # Si lo obtienes de un DataFrame
    data = subject
    x = np.arange(len(data))
    
    # Crear 'size' puntos equidistantes
    x_new = np.linspace(0, len(data) - 1, size*(len(data)-1) + len(data))
    
    if method == 'lineal':
        data_interp = np.interp(x_new, x, data)
    elif method == 'herm':
        interpolator = PchipInterpolator(x, data)
        data_interp = interpolator(x_new)
    
    return x_new, data_interp

nombres = ['A','B','C','D','E','F','H','I','J']

# for i in range(len(nombres)):
#     # Nombre del archivo CSV
#     archivo_csv = 's' + nombres[i] +'.csv'

#     # Leer el archivo
#     df = pd.read_csv(archivo_csv)

#     # Número de columnas en el DataFrame (menos la primera)
#     num_graficas = len(df.columns) - 1

#     # Configurar filas y columnas para los subplots
#     filas = (num_graficas + 1) // 2  # Asegura suficientes filas para dos columnas
#     fig, axes = plt.subplots(filas, 2, figsize=(12, 4 * filas))  # Ajusta el tamaño de la figura

#     # Aplanar los ejes para iterar fácilmente
#     axes = axes.flatten()

#     # Iterar sobre las columnas (omitiendo la primera)
#     for i, columna_a_graficar in enumerate(df.columns[1:]):
#         axes[i].scatter(range(len(df[columna_a_graficar])), df[columna_a_graficar], marker='.', alpha=0.7)
#         axes[i].set_title(f'Gráfica de {columna_a_graficar}')
#         axes[i].set_xlabel('Índice')
#         axes[i].set_ylabel('Valores')

#     # Ocultar los subplots vacíos (si hay más subplots que columnas)
#     for j in range(i + 1, len(axes)):
#         fig.delaxes(axes[j])

#     # Ajustar el diseño para que no se superpongan las etiquetas
#     plt.tight_layout()
#     plt.show()

#     J_por_columna = [J_univariante(df[columna],1,False) for columna in df.columns[1:]]

#     for i in range(len(J_por_columna)):
#         print('Columna: ', df.columns[i+1],', J index: ', J_por_columna[i])


#     # Número de columnas
#     num_graficas = len(df.columns[1:])
#     filas = (num_graficas + 1) // 2  # Número de filas (ajustado para 2 columnas)

#     # Crear subplots
#     fig, axes = plt.subplots(filas, 2, figsize=(11, 4 * filas))  # Ajusta el tamaño de la figura

#     # Aplanar los ejes para iterar fácilmente
#     axes = axes.flatten()

#     # Iterar sobre las columnas y graficar
#     for i, columna in enumerate(df.columns[1:]):
#         fases = np.angle(np.fft.rfft(df[columna]))

#         # Graficar en el subplot correspondiente
#         axes[i].scatter(range(len(fases)), fases, marker='.')
#         axes[i].set_title(f'Fases de {columna}')
#         axes[i].set_xlabel('Frecuencia')
#         axes[i].set_ylabel('Ángulo (radianes)')

#     # Ocultar los ejes vacíos (si los hay)
#     for j in range(i + 1, len(axes)):
#         fig.delaxes(axes[j])

#     # Ajustar el diseño
#     plt.tight_layout()
#     plt.show()



In [3]:
J_min = np.load('J_minus_continuo.npy')

In [13]:
# Cargar J mínimo continuo
J_min = np.load('J_minus_continuo.npy')

# Archivos y columnas válidas
nombres = ['A','B','C','D','E','F','H','I','J']
columnas_validas = ["PPn", "RRn", "TTn", "PR", "RT", "PT", "TPn"]

# Procesar cada archivo
for letra in nombres:
    archivo_csv = f"s{letra}.csv"
    df = pd.read_csv(archivo_csv)
    columnas = [col for col in df.columns if col in columnas_validas]

    # Calcular J_univariante
    J_uni = {}
    for col in columnas:
        serie = df[col].dropna().values
        J_uni[col] = J_univariante(serie, tau=1, corte=False)

    # Calcular J_bivariante
    J_bi = {}
    for col1, col2 in itertools.combinations(columnas, 2):
        serie1 = df[col1].dropna().values
        serie2 = df[col2].dropna().values
        J_bi[f"{col1}-{col2}"] = J_bivariante(serie1, serie2, corte=False)

    # Crear etiquetas y valores
    etiquetas = list(J_uni.keys()) + list(J_bi.keys())
    valores = list(J_uni.values()) + list(J_bi.values())
    colores = ['skyblue'] * len(J_uni) + ['salmon'] * len(J_bi)

    # Calcular los valores de referencia J mínimo continuo
    numeros = []
    for col in J_uni:
        N = len(df[col].dropna())
        mitad = N // 2
        idx = np.where(J_min[0, :] == mitad)[0]
        numeros.append(J_min[1, idx[0]] if idx.size > 0 else np.nan)

    for par in J_bi:
        col1, col2 = par.split('-')
        N = min(len(df[col1].dropna()), len(df[col2].dropna()))
        mitad = N // 2
        idx = np.where(J_min[0, :] == mitad)[0]
        numeros.append(J_min[1, idx[0]] if idx.size > 0 else np.nan)

    # Crear gráfico
    x = np.arange(len(etiquetas))
    plt.figure(figsize=(14, 6))
    plt.bar(x, valores, color=colores)
    plt.plot(x, numeros, 'ko', label='J mínimo continuo')
    plt.xticks(x, etiquetas, rotation=90)
    plt.title(f"Índice J - Hoja {letra}")
    plt.ylabel("Valor de J")
    plt.legend(handles=[
        plt.Line2D([0], [0], color='skyblue', lw=4, label='J_univariante'),
        plt.Line2D([0], [0], color='salmon', lw=4, label='J_bivariante'),
        plt.Line2D([0], [0], marker='o', color='k', lw=0, label='J mínimo continuo')
    ])
    plt.tight_layout()
    plt.savefig(f"J_indices_{letra}.png")
    plt.close()

    print(f"[✓] Procesado: {archivo_csv}")

[✓] Procesado: sA.csv
[✓] Procesado: sB.csv
[✓] Procesado: sC.csv
[✓] Procesado: sD.csv
[✓] Procesado: sE.csv
[✓] Procesado: sF.csv
[✓] Procesado: sH.csv
[✓] Procesado: sI.csv
[✓] Procesado: sJ.csv


In [17]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Parámetros configurables
MS = 1.0         # Puedes cambiar esto
indice_gamma = 1 # Puedes cambiar esto (debe cumplir 1 ≤ indice_gamma ≤ 6)

def calcular_gamma_opt(data, gamma_index, MS):
    N = len(data)
    sd = np.std(data, ddof=1)
    eps = sd / MS
    maxdat = np.max(data)
    data = np.concatenate(([0.0], data, [maxdat + 100 * eps]))  # data[0], data[N+1]

    Ci = [0] * (gamma_index + 2)  # Necesitamos Ci[i-1], Ci[i], Ci[i+1]

    for j in range(1, N + 1):
        for i in range(1, j):
            k = 0
            while k <= gamma_index + 1 and abs(data[i + k] - data[j + k]) <= eps:
                if k in (gamma_index - 1, gamma_index, gamma_index + 1):
                    Ci[k] += 1
                k += 1

    norm = 2.0 / (N * (N - 1))
    C = [1.0] + [0.0] * (gamma_index + 1)
    for k in (gamma_index - 1, gamma_index, gamma_index + 1):
        if k >= 1:
            C[k] = Ci[k] * norm

    denominator = C[gamma_index - 1] * C[gamma_index + 1]
    gamma = 0.0
    if denominator != 0:
        gamma = 1.0 - (C[gamma_index] ** 2) / denominator

    return gamma

# Archivos a procesar
nombres = ['A','B','C','D','E','F','H','I','J']
columnas_validas = ["PPn", "RRn", "TTn", "PR", "RT", "PT", "TPn"]

for letra in nombres:
    archivo_csv = f"s{letra}.csv"
    df = pd.read_csv(archivo_csv)
    columnas = [col for col in df.columns if col in columnas_validas]

    gamma_valores = {}
    for col in columnas:
        serie = df[col].dropna().values
        gamma_valores[col] = calcular_gamma_opt(serie, indice_gamma, MS)

    # Gráfica
    etiquetas = list(gamma_valores.keys())
    valores = list(gamma_valores.values())

    plt.figure(figsize=(10, 5))
    plt.bar(etiquetas, valores, color='steelblue')
    plt.title(f"Índice gamma[{indice_gamma}] - Hoja {letra}")
    plt.ylabel(f"gamma[{indice_gamma}]")
    plt.tight_layout()
    plt.savefig(f"gamma_{indice_gamma}_{letra}.png")
    plt.close()

    print(f"[✓] Procesado: {archivo_csv}")


[✓] Procesado: sA.csv
[✓] Procesado: sB.csv
[✓] Procesado: sC.csv
[✓] Procesado: sD.csv
[✓] Procesado: sE.csv
[✓] Procesado: sF.csv
[✓] Procesado: sH.csv
[✓] Procesado: sI.csv
[✓] Procesado: sJ.csv


In [2]:
def calcular_gamma_opt(data, gamma_index, MS):
    N = len(data)
    sd = np.std(data, ddof=1)
    eps = sd / MS
    maxdat = np.max(data)
    data = np.concatenate(([0.0], data, [maxdat + 100 * eps]))  # data[0], data[N+1]

    Ci = [0] * (gamma_index + 2)  # Necesitamos Ci[i-1], Ci[i], Ci[i+1]

    for j in range(1, N + 1):
        for i in range(1, j):
            k = 0
            while k <= gamma_index + 1 and abs(data[i + k] - data[j + k]) <= eps:
                if k in (gamma_index - 1, gamma_index, gamma_index + 1):
                    Ci[k] += 1
                k += 1

    norm = 2.0 / (N * (N - 1))
    C = [1.0] + [0.0] * (gamma_index + 1)
    for k in (gamma_index - 1, gamma_index, gamma_index + 1):
        if k >= 1:
            C[k] = Ci[k] * norm

    denominator = C[gamma_index - 1] * C[gamma_index + 1]
    gamma = 0.0
    if denominator != 0:
        gamma = 1.0 - (C[gamma_index] ** 2) / denominator

    return gamma

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import itertools
from scipy.stats import pearsonr

# Parámetros configurables
usar_archivos_guardados = False  # Cambia a True si quieres usar los .npy guardados
method = 'lineal'                  # 'herm' o 'lineal'
size = 2                         # 0 = sin interpolación
MS = 1.0                         # Parámetro para gamma
indice_gamma = 1                 # Entre 1 y 6

# Configuración general
nombres = ['A', 'B', 'C', 'D', 'E', 'F', 'H', 'I', 'J']
columnas_validas = ["PPn", "RRn", "TTn", "PR", "RT", "PT", "TPn"]
ruta_datos = "datos"
os.makedirs(ruta_datos, exist_ok=True)

# Cargar J mínimo continuo
J_min = np.load('J_minus_continuo.npy')

# Diccionarios para almacenar todos los resultados
resultados_J = {}
resultados_gamma = {}

# Calcular o cargar datos
for letra in nombres:
    archivo_J = f"{ruta_datos}/J_{letra}_method-{method}_size-{size}.npy"
    archivo_gamma = f"{ruta_datos}/gamma_{letra}_gamma-{indice_gamma}_MS-{MS}.npy"

    if usar_archivos_guardados and os.path.exists(archivo_J) and os.path.exists(archivo_gamma):
        J_uni_bi = np.load(archivo_J, allow_pickle=True).item()
        gamma_vals = np.load(archivo_gamma, allow_pickle=True).item()
    else:
        df = pd.read_csv(f"s{letra}.csv")
        columnas = [col for col in df.columns if col in columnas_validas]

        # Calcular J_univariante
        J_uni_bi = {}
        for col in columnas:
            serie = df[col].dropna().values
            if size > 0:
                _, serie = interpolador(serie, method, size)
            J_uni_bi[col] = J_univariante(serie, tau=1, corte=False)

        # Calcular J_bivariante
        for col1, col2 in itertools.combinations(columnas, 2):
            serie1 = df[col1].dropna().values
            serie2 = df[col2].dropna().values
            if size > 0:
                _, serie1 = interpolador(serie1, method, size)
                _, serie2 = interpolador(serie2, method, size)
            J_uni_bi[f"{col1}-{col2}"] = J_bivariante(serie1, serie2, corte=False)

        # Calcular gamma
        gamma_vals = {}
        for col in columnas:
            serie = df[col].dropna().values
            gamma_vals[col] = calcular_gamma_opt(serie, indice_gamma, MS)

        np.save(archivo_J, J_uni_bi)
        np.save(archivo_gamma, gamma_vals)

    resultados_J[letra] = J_uni_bi
    resultados_gamma[letra] = gamma_vals

    # Gráfica del índice J
    etiquetas = list(J_uni_bi.keys())
    valores = list(J_uni_bi.values())
    colores = ['skyblue' if '-' not in e else 'salmon' for e in etiquetas]

    # Valores de referencia J mínimo continuo
    numeros = []
    for etiqueta in etiquetas:
        if '-' in etiqueta:
            col1, col2 = etiqueta.split('-')
            N = min(len(df[col1].dropna()), len(df[col2].dropna()))
        else:
            N = len(df[etiqueta].dropna())
        mitad = N // 2
        idx = np.where(J_min[0, :] == mitad)[0]
        numeros.append(J_min[1, idx[0]] if idx.size > 0 else np.nan)

    # Gráfica
    x = np.arange(len(etiquetas))
    ymin = min(np.nanmin(valores), np.nanmin(numeros))
    plt.figure(figsize=(14, 6))
    plt.bar(x, valores, color=colores)
    plt.plot(x, numeros, 'ko', label='J mínimo continuo')
    plt.xticks(x, etiquetas, rotation=90)
    plt.ylim([ymin, 1.0])
    plt.title(f"Índice J - Hoja {letra}")
    plt.ylabel("Valor de J")
    plt.legend(handles=[
        plt.Line2D([0], [0], color='skyblue', lw=4, label='J_univariante'),
        plt.Line2D([0], [0], color='salmon', lw=4, label='J_bivariante'),
        plt.Line2D([0], [0], marker='o', color='k', lw=0, label='J mínimo continuo')
    ])
    plt.tight_layout()
    plt.savefig(f"J_indices_{letra}.png")
    plt.close()

    # Gráfica del índice gamma
    etiquetas_gamma = list(gamma_vals.keys())
    valores_gamma = list(gamma_vals.values())
    plt.figure(figsize=(10, 5))
    plt.bar(etiquetas_gamma, valores_gamma, color='steelblue')
    plt.title(f"Índice gamma[{indice_gamma}] - Hoja {letra}")
    plt.ylabel(f"gamma[{indice_gamma}]")
    plt.tight_layout()
    plt.savefig(f"gamma_{indice_gamma}_{letra}.png")
    plt.close()

    print(f"[✓] Procesado: Hoja {letra}")

# Correlaciones por columna
corrs_col = {}
for col in columnas_validas:
    Js = []
    gammas = []
    for letra in nombres:
        j_dict = resultados_J[letra]
        g_dict = resultados_gamma[letra]
        if col in j_dict and col in g_dict:
            Js.append(j_dict[col])
            gammas.append(g_dict[col])
    if len(Js) >= 2:
        corrs_col[col] = pearsonr(Js, gammas)[0]

# Graficar correlaciones por columna
plt.figure(figsize=(10, 5))
plt.bar(corrs_col.keys(), corrs_col.values(), color='mediumseagreen')
plt.title(f"Correlación Pearson por columna\nmethod={method}, size={size}, gamma={indice_gamma}, MS={MS}")
plt.ylabel("Coeficiente de correlación")
plt.tight_layout()
plt.savefig("correlacion_por_columna.png")
plt.close()

# Correlaciones por paciente
corrs_paciente = {}
for letra in nombres:
    j_dict = resultados_J[letra]
    g_dict = resultados_gamma[letra]
    comunes = [col for col in columnas_validas if col in j_dict and col in g_dict]
    if len(comunes) >= 2:
        Js = [j_dict[col] for col in comunes]
        gammas = [g_dict[col] for col in comunes]
        corrs_paciente[letra] = pearsonr(Js, gammas)[0]

# Graficar correlaciones por paciente
plt.figure(figsize=(10, 5))
plt.bar(corrs_paciente.keys(), corrs_paciente.values(), color='tomato')
plt.title(f"Correlación Pearson por paciente\nmethod={method}, size={size}, gamma={indice_gamma}, MS={MS}")
plt.ylabel("Coeficiente de correlación")
plt.tight_layout()
plt.savefig("correlacion_por_paciente.png")
plt.close()


KeyboardInterrupt: 